In [1]:
import pandas as pd
from llama_index.core.evaluation import DatasetGenerator, RelevancyEvaluator
from llama_index.core import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chat_models import ChatOpenAI

import nest_asyncio
nest_asyncio.apply()

import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")


In [2]:
def load_documents_from_directory(directory_path):
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith('.txt'):  # adjust file extension as needed
            with open(os.path.join(directory_path, filename), 'r', encoding='utf-8') as file:
                documents.append(file.read())
    return documents

# Function to perform semantic chunking 
def semantic_chunking(text, model_name,breakpoint_threshold_type, breakpoint_threshold_amount):
    embed_model = HuggingFaceEmbeddings(model_name=model_name)
    semantic_chunker = SemanticChunker(embed_model, 
                                       breakpoint_threshold_type=breakpoint_threshold_type,
                                       breakpoint_threshold_amount = breakpoint_threshold_amount)#,sentence_split_regex=r"[.?!]\s*"
    
    docs = semantic_chunker.create_documents(text)

    # Convert Langchain documents to LlamaIndex documents
    llama_docs = [Document(text=doc.page_content) for doc in docs]
    return llama_docs

#vectordb creation and store the chunks
def create_vector_db(model_name,chunks,persist_directory,):
    # Define the directory where ChromaDB will store data
    persist_directory = persist_directory

    # Initialize the embedding function with trust_remote_code=True
    embedding_function = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"trust_remote_code": True}  # This allows loading custom model code
    )

    # Create ChromaDB instance with the embedding function
    vectordb = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_function
    )

    # Alternative Option 2: If you want to preserve metadata
    vectordb.add_texts(
        texts=[chunk.get_content() for chunk in chunks],
        metadatas=[chunk.metadata for chunk in chunks]
    )
    
    # Persist the database locally
    vectordb.persist()

    print(f"ChromaDB has been successfully saved at: {persist_directory}")

#loading created vector db
def load_vector_db(model_name, persist_directory):

    # Initialize the embedding function with trust_remote_code=True
    embedding_function = HuggingFaceEmbeddings(
        model_name=model_name,
        model_kwargs={"trust_remote_code": True}  # This allows loading custom model code
    )

    # Load the existing ChromaDB instance
    vectordb = Chroma(
        persist_directory=persist_directory,
        embedding_function=embedding_function
    )

    print(f"ChromaDB has been successfully loaded from: {persist_directory}")
    return vectordb

def get_similar_chunks(question,vectordb):
    similar_chunks = vectordb.similarity_search(question,k=10)
    return similar_chunks

def get_answer(model_name,query,similar_chunks):

    prompt_template = PromptTemplate(
        input_variables=['query', 'context'],
        template="""
        You are an expert assistant specializing in question-answering. The provided context will be from religious texts such as holy books, stories, prayers, or research papers.

        Your task is to carefully read, understand, and analyze the given context before responding to the user's questions. Ensure your answers are accurate, concise, and human-friendly, while remaining clear and to the point.

        Important:
        Do not make assumptions or fabricate responses.

        Based on the context provided below, answer the question:

        Context: {context}

        Question: {question}

        Answer:
        """
    )


    llm = ChatOpenAI(model=model_name,temperature=0,max_tokens=1500)#
    chain = LLMChain(llm=llm, prompt=prompt_template)

    return chain.run(question=query, context=similar_chunks)


In [8]:
def load_text_as_is(file_path):
    """
    Load the content of a file and return it exactly as it appears (preserving formatting).

    Args:
        file_path (str): Path to the .txt file.

    Returns:
        str: The content of the file as a single string, preserving its original format.
    """
    with open(file_path, 'r', encoding='utf-8') as file:
        content = file.read()  # Read the entire file content as a string
    return content


file_path = r"D:\Anand\ra-granth\text_files\text_files_chunked\Chunks of 005A Kamandakya - Nitisara english - extracted_text.txt"
content = load_text_as_is(file_path)
print(content)
print("Length of content:",len(content))

[
  '\n\n\n\n\n\n\n\n\n\n\n\n\nKAMANDAKIYA NITISARA\nOR\nTHE ELEMENTS OF POLITY\n(IN ENGLISH.)\nEDITED AND PUBLISHED BY\nMANMATHA NATH DUTT, M.A., M.R.A.S. RECTOR, KESHUB ACADEMY ;\nuthor of the English Translations of the Ramayana, Srtmadbhaga-\nvatam, Vishnupuranam, Mahabharata, Bhagavat-Gita\nand other works.',
  'CALCUTTA:\nPRINTED BY II.',
  'C.',
  'DASS,\nELYSIUM PRESS, 65/2 BEADON STREET,\n-\n1896,\n\n\nCARPENTIER\n\n\nJC50\nK2513\n1896\nINTRODUCTION.',
  'MAIN\no:\nTHE superiority of the ancient Hindus in metaphysical and\ntheological disquisitions has been established beyond all\ndoubts.',
  'Our literature abounds in trea-\nThe science of Polity: its\norigin.',
  'tises, which, for philosophical discus-\nsions, sound reasonings and subtle\ninferences regarding many momentous problems of existence, have\nnot been beaten down by the modern age of culture and enlighten-\nment.',
  'The world has all along been considered by the ancient\nHindu writers as a flood-gate of miseries

In [ ]:
from llama_index.core.evaluation import RelevancyEvaluator

def evaluate_answer(query, similar_chunks, answer):
    """
    Evaluate the relevance of the answer against the provided context and query.

    Args:
        query (str): The input question.
        similar_chunks (list of Document): The retrieved similar chunks.
        answer (str): The generated answer.

    Returns:
        float: The relevance score.
    """
    # Extract the page content from similar_chunks as a list of strings
    contexts = [chunk.page_content for chunk in similar_chunks]

    evaluator = RelevancyEvaluator()
    # Pass the correct arguments to the evaluator
    relevance_score = evaluator.evaluate(query=query, contexts=contexts, response=answer)
    return relevance_score



In [ ]:
# 1> evaluation
def main(question):
    persist_directory = f"D:\\Anand\\ra-granth\\test_db\\chroma_db_o.7"
    embed_model_name = "Alibaba-NLP/gte-base-en-v1.5" 

    # Load vector DB
    vectordb = load_vector_db(embed_model_name, persist_directory)

    model_name = "gpt-4o-mini"
    similar_chunks = get_similar_chunks(question, vectordb)
    
    ansr = get_answer(model_name,question,similar_chunks)

    print("Answer:")
    print(ansr)
   
    evaluation_result = evaluate_answer(question,similar_chunks,ansr)  
      
    print("Answer : ",evaluation_result.response) #The generated response that was evaluated.
    print("passing : ",evaluation_result.passing) #Indicates whether the generated response was considered relevant and accurate based on the evaluation criteria. A value of True means the evaluation passed.
    print("feedback : ",evaluation_result.feedback) #Additional feedback about the evaluation. 'YES' suggests the response met the expected standards.
    print("score : ",evaluation_result.score) #A numerical score indicating the relevance or correctness of the response. Higher scores (e.g., 1.0) imply a high-quality response.
    print("pairwise_source : ",evaluation_result.pairwise_source) #Used for pairwise comparison evaluations. In this case, no pairwise comparison was performed, so it is None
    print("invalid_result : ",evaluation_result.invalid_result) # Indicates whether the evaluation result is invalid. False means the evaluation is valid.
    print("invalid_reason : ",evaluation_result.invalid_reason) # If the result were invalid, this would provide the reason. Since invalid_result is False, this is None.

question = "What is the profession of Manmatha Nath Dutt, aside from being an editor and publisher? what are their translations?"
main(question)

ChromaDB has been successfully loaded from: D:\Anand\ra-granth\test_db\chroma_db_o.7
Answer:
Manmatha Nath Dutt was also a rector at Keshub Academy. Aside from his roles as an editor and publisher, he is known for his English translations of several significant religious texts, including the Ramayana, Srimad Bhagavatam, Vishnu Purana, Mahabharata, and Bhagavad Gita.
Answer :  Manmatha Nath Dutt was also a rector at Keshub Academy. Aside from his roles as an editor and publisher, he is known for his English translations of several significant religious texts, including the Ramayana, Srimad Bhagavatam, Vishnu Purana, Mahabharata, and Bhagavad Gita.
passing :  True
feedback :  YES
score :  1.0
pairwise_source :  None
invalid_result :  False
invalid_reason :  None


In [14]:
def get_answer_with_retries(model_name, query, similar_chunks, retry_limit,score_thresold):
    """
    Get an answer with retries if the response is not satisfactory.

    Args:
        model_name (str): Name of the LLM to use.
        query (str): The input query.
        similar_chunks (list): Retrieved chunks for context.
        retry_limit (int): Number of retry attempts.

    Returns:
        str: The best answer generated by the LLM.
    """
    prompt_template = PromptTemplate(
        input_variables=['query', 'context'],
        template="""
        You are an expert assistant specializing in question-answering. The provided context will be from religious texts such as holy books, stories, prayers, or research papers.

        Your task is to carefully read, understand, and analyze the given context before responding to the user's questions. Ensure your answers are accurate, concise, and human-friendly, while remaining clear and to the point.

        Important:
        Do not make assumptions or fabricate responses.

        Based on the context provided below, answer the question:

        Context: {context}

        Question: {query}

        Answer:
        """
    )

    llm = ChatOpenAI(model=model_name, temperature=0, max_tokens=1500)
    chain = LLMChain(llm=llm, prompt=prompt_template)

    retries = 0
    while retries < retry_limit:
        # Combine similar chunks into a single string
        context = "\n".join([chunk.page_content for chunk in similar_chunks])

        # Generate answer
        try:
            answer = chain.run(query=query, context=context)
        except ValueError as e:
            print(f"Error during chain execution: {e}")
            return "Failed to generate an answer."

        # Evaluate answer
        evaluation_result = evaluate_answer(query, similar_chunks, answer)
        score = evaluation_result.score  # Extract score from the EvaluationResult

        if score >= score_thresold:  # Assume a score threshold of 0.8
            return answer
        else:
            retries += 1
            print(f"Retry {retries}: Re-evaluating the answer...")

    return "Failed to generate a satisfactory answer after retries."


In [ ]:
# 2> evaluate and regenerate the answer(if thresold is not met)
def main():
    persist_directory = f"D:\\Anand\\ra-granth\\test_db\\chroma_db_o.7"
    model_name = "Alibaba-NLP/gte-base-en-v1.5" 

    # Load vector DB
    vectordb = load_vector_db(model_name, persist_directory)

    question = "What is the profession of Manmatha Nath Dutt, aside from being an editor and publisher? what are their translations?"
    similar_chunks = get_similar_chunks(question, vectordb)

    # Get answer with self-correcting logic
    ansr = get_answer_with_retries("gpt-4o-mini", question, similar_chunks , retry_limit, score_thresold)
    print("Answer:")
    print(ansr)


In [16]:
model_name = "gpt-4o-mini"
retry_limit = 5
score_thresold = 0.9

if __name__ == "__main__":
    main()


ChromaDB has been successfully loaded from: D:\Anand\ra-granth\test_db\chroma_db_o.7
Answer:
Manmatha Nath Dutt is also a rector at Keshub Academy. He is known for his English translations of several significant religious texts, including the Ramayana, Srimad Bhagavatam, Vishnu Purana, Mahabharata, and Bhagavad Gita.
